In [54]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

In [9]:
# Descarga del dataset
!wget --no-check-certificate -q https://www.mecon.gob.ar/dataset/precios-internacionales-minerales.xlsx

In [10]:
df: pd.DataFrame = pd.read_excel("precios-internacionales-minerales.xlsx")

In [11]:
df.sample(10)

,Año,Mes,Fecha,Precio,Mineral,Unidad de medida,Numero Índice
3449,2023,2,2023M02,1854.54,Oro (Au),Dólar por Onza Troy (USD/troy oz),143.568028
3800,2016,1,2016M01,1646.20,Plomo (Pb),Dólar por Tonelada Métrica (USDT/TM),82.427872
2520,2018,5,2018M05,904.73,Platino (Pt),Dólar por Onza Troy (USD/troy oz),112.142246
3929,1990,6,1990M06,11.00,Uranio (U),Dólar por Libra (USD/lb),38.314176
4216,2014,5,2014M05,28.54,Uranio (U),Dólar por Libra (USD/lb),99.407872
2580,2023,5,2023M05,1062.73,Platino (Pt),Dólar por Onza Troy (USD/troy oz),131.726514
472,1993,1,1993M01,2256.85,Cobre (Cu),Dólar por Tonelada Métrica (USD/t),37.999865
1993,2010,10,2010M10,2372.14,Zinc (Zn),Dólar por Tonelada Métrica (USD/t),92.311943
1387,1996,8,1996M08,7054.36,Níquel,Dólar por Tonelada Métrica (USD/t),61.219343
2838,2008,7,2008M07,18.03,Plata (Ag),Dólar por Onza Troy (USD/troy oz),115.428937


No hay nulos

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4886 entries, 0 to 4885
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Año               4886 non-null   int64  
 1   Mes               4886 non-null   int64  
 2   Fecha             4886 non-null   object 
 3   Precio            4886 non-null   float64
 4   Mineral           4886 non-null   object 
 5   Unidad de medida  4886 non-null   object 
 6   Numero Índice     4886 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 267.3+ KB


Chequeo de Unidad de medida del oro

In [13]:
df['Mineral'].unique()

array(['Aluminio (Al)', 'Cobre (Cu)', 'Estaño (Sn)', 'Níquel',
       'Zinc (Zn)', 'Platino (Pt)', 'Plata (Ag)', 'Oro (Au)',
       'Plomo (Pb)', 'Uranio (U)', 'Carbonato de Litio 99,5% (Li2CO3)',
       'Hierro (Fe)'], dtype=object)

In [53]:
df_oro = df[df['Mineral'] == 'Oro (Au)']

In [55]:
df_oro['Unidad de medida'].unique()

array(['Dólar por Onza Troy (USD/troy oz)'], dtype=object)

Ordenar Dataset por fecha

- Vemos que el dataset tiene datos mensuales desde enero 1990 a abril de 2026

In [56]:
df_oro.sort_values(by=["Año", "Mes"], ascending=False)

,Año,Mes,Fecha,Precio,Mineral,Unidad de medida,Numero Índice
3487,2026,4,2026M04,4721.42,Oro (Au),Dólar por Onza Troy (USD/troy oz),365.505709
3486,2026,3,2026M03,4855.54,Oro (Au),Dólar por Onza Troy (USD/troy oz),375.888523
3485,2026,2,2026M02,5019.97,Oro (Au),Dólar por Onza Troy (USD/troy oz),388.617767
3484,2026,1,2026M01,4752.75,Oro (Au),Dólar por Onza Troy (USD/troy oz),367.931101
3483,2025,12,2025M12,4309.23,Oro (Au),Dólar por Onza Troy (USD/troy oz),333.596284
...,...,...,...,...,...,...,...
3056,1990,5,1990M05,369.05,Oro (Au),Dólar por Onza Troy (USD/troy oz),28.569770
3055,1990,4,1990M04,374.24,Oro (Au),Dólar por Onza Troy (USD/troy oz),28.971550
3054,1990,3,1990M03,393.06,Oro (Au),Dólar por Onza Troy (USD/troy oz),30.428488
3053,1990,2,1990M02,416.81,Oro (Au),Dólar por Onza Troy (USD/troy oz),32.267080


chequeo que estén todos los años y tengan una cantidad de muestras uniformes.

In [58]:
df_oro'Año'].value_counts().sort_index(ascending=False)

SyntaxError: unmatched ']' (997983568.py, line 1)

Chequeo que estén todos los meses

In [41]:
meses_por_anio = df.groupby('Año')['Mes'].nunique()

# Mostrar años que no tienen 12 meses
meses_faltantes_por_anio = meses_por_anio[meses_por_anio != 12]

if not meses_faltantes_por_anio.empty:
    print("Años con meses incompletos:")
    display(meses_faltantes_por_anio)
else:
    print("Todos los años tienen los 12 meses.")

# También mostrar el conteo de meses para todos los años para ver el panorama completo
print("\nConteo de meses por año:")
display(meses_por_anio)

Años con meses incompletos:


,Mes
Año,
2026,4



Conteo de meses por año:


,Mes
Año,
1990,12
1991,12
1992,12
1993,12
1994,12
1995,12
1996,12
1997,12
1998,12


No hay duplicados

In [61]:
df.duplicated().sum()

np.int64(0)

### Detección de outliers

In [59]:
px.box(df, df_oro['Año'], df_oro['Precio'], title = "Boxplot del precio del oro en función del año")

In [62]:
print("\n. Control de continuidad temporal (Huecos en la serie por mineral):")
# Este paso es clave para el informe: justifica si la serie es continua o necesita imputación
minerales = df['Mineral'].unique()

for mineral in minerales:
    df_min = df[df['Mineral'] == mineral]

    # Identificamos los límites temporales del mineral
    min_year = df_min['Año'].min()
    max_year = df_min['Año'].max()

    min_month = df_min[df_min['Año'] == min_year]['Mes'].min()
    max_month = df_min[df_min['Año'] == max_year]['Mes'].max()

    # Generamos el conjunto de pares (Año, Mes) que deberíamos tener idealmente
    pares_esperados = set()
    for y in range(min_year, max_year + 1):
        m_inicio = min_month if y == min_year else 1
        m_fin = max_month if y == max_year else 12
        for m in range(m_inicio, m_fin + 1):
            pares_esperados.add((y, m))

    # Obtenemos los pares (Año, Mes) que realmente existen en el dataset
    pares_actuales = set(zip(df_min['Año'], df_min['Mes']))

    # Comparamos para ver qué falta
    faltantes = pares_esperados - pares_actuales

    if len(faltantes) > 0:
        print(f"  ⚠️ {mineral}: Faltan {len(faltantes)} meses en la serie histórica.")
        # Opcional: print(f"     Meses faltantes: {sorted(list(faltantes))}")
    else:
        print(f"  ✅ {mineral}: Serie temporal completa, sin saltos.")

# 4. Resumen estadístico multivariado (Ideal para copiar al informe)
print("\n--- ESTADÍSTICAS DESCRIPTIVAS ---")
display(df.groupby('Mineral')[['Precio', 'Numero Índice']].describe())

# ---------------------------------------------------------
# EXPORTACIÓN DEL DATASET LIMPIO
# ---------------------------------------------------------
archivo_limpio = 'dataset_minerales_limpio.csv'
df.to_csv(archivo_limpio, index=False)
print(f"\n✅ Archivo resultante exportado con éxito: {archivo_limpio}")


3. Control de continuidad temporal (Huecos en la serie por mineral):
  ✅ Aluminio (Al): Serie temporal completa, sin saltos.
  ✅ Cobre (Cu): Serie temporal completa, sin saltos.
  ✅ Estaño (Sn): Serie temporal completa, sin saltos.
  ✅ Níquel: Serie temporal completa, sin saltos.
  ✅ Zinc (Zn): Serie temporal completa, sin saltos.
  ✅ Platino (Pt): Serie temporal completa, sin saltos.
  ✅ Plata (Ag): Serie temporal completa, sin saltos.
  ✅ Oro (Au): Serie temporal completa, sin saltos.
  ✅ Plomo (Pb): Serie temporal completa, sin saltos.
  ✅ Uranio (U): Serie temporal completa, sin saltos.
  ✅ Carbonato de Litio 99,5% (Li2CO3): Serie temporal completa, sin saltos.
  ✅ Hierro (Fe): Serie temporal completa, sin saltos.

--- ESTADÍSTICAS DESCRIPTIVAS ---


Precio                                       \
                                   count          mean           std      min   
Mineral                                                                         
Aluminio (Al)                      436.0   1871.902844    498.425070  1039.81   
Carbonato de Litio 99,5% (Li2CO3)   90.0  23290.515000  22355.256621  6750.00   
Cobre (Cu)                         436.0   5098.515826   2904.509438  1377.28   
Estaño (Sn)                        436.0  14842.703578  10057.763470  3694.50   
Hierro (Fe)                        436.0     75.276170     47.975418    26.47   
Níquel                             436.0  13658.358968   7578.318535  3871.93   
Oro (Au)                           436.0   1017.152317    821.712990   256.08   
Plata (Ag)                         436.0     14.681904     11.921159     3.65   
Platino (Pt)                       436.0    896.195917    444.951951   341.19   
Plomo (Pb)                         436.0   1414.860849    780.734436   375.70   
Uranio (U)                         436.0     31.954518     24.628982     7.10   
Zinc (Zn)                          436.0   1879.764885    858.112087   747.60   

                                                                               \
                                         25%        50%         75%       max   
Mineral                                                                         
Aluminio (Al)                      1484.9625   1776.790   2190.9125   3599.85   
Carbonato de Litio 99,5% (Li2CO3)  9451.5025  13000.000  25926.1375  80909.09   
Cobre (Cu)                         2245.9325   5154.965   7549.9775  13012.00   
Estaño (Sn)                        5705.6750  14119.880  21206.2475  49538.44   
Hierro (Fe)                          31.0000     65.000    107.4625    214.43   
Níquel                             7728.2675  12659.985  17309.3700  52179.05   
Oro (Au)                            366.0850    811.355   1396.1350   5019.97   
Plata (Ag)                            5.0700     13.370     19.8825     92.06   
Platino (Pt)                        436.5175    899.095   1169.3375   2433.83   
Plomo (Pb)                          584.2250   1681.430   2083.9300   3719.72   
Uranio (U)                           10.4300     25.660     44.4825    136.22   
Zinc (Zn)                          1061.5500   1841.450   2504.6600   4405.40   

                                  Numero Índice                          \
                                          count        mean         std   
Mineral                                                                   
Aluminio (Al)                             436.0  100.980884   26.887829   
Carbonato de Litio 99,5% (Li2CO3)          90.0  172.522333  165.594493   
Cobre (Cu)                                436.0   85.846607   48.904875   
Estaño (Sn)                               436.0   72.552962   49.163586   
Hierro (Fe)                               436.0   98.839509   62.992934   
Níquel                                    436.0  118.530351   65.766375   
Oro (Au)                                  436.0   78.742196   63.612386   
Plata (Ag)                                436.0   93.994262   76.319843   
Platino (Pt)                              436.0  111.084438   55.152268   
Plomo (Pb)                                436.0   70.844350   39.092624   
Uranio (U)                                436.0  111.301004   85.785378   
Zinc (Zn)                                 436.0   73.151142   33.393473   

                                                                     \
                                         min        25%         50%   
Mineral                                                               
Aluminio (Al)                      56.093153  80.107163   95.849967   
Carbonato de Litio 99,5% (Li2CO3)  50.000000  70.011130   96.296296   
Cobre (Cu)                         23.190046  37.816041   86.797074   
Estaño (Sn)                        18.059171  27.8


✅ Archivo resultante exportado con éxito: dataset_minerales_limpio.csv


El dataset resultante, es el mismo que el original.